<a href="https://colab.research.google.com/github/cintiapinho/Aulas_FATEC_PLN/blob/main/Materiais/Aula_5_6_ExtracaoCaracteristicas_ML_em_PLN/notebook_exercicios.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sessões 05 e 06 — Exercícios (entrega)

**Processamento de Linguagem Natural**

Atividade prática

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

---

## Desafio — comparando os três métodos numa base bem mais suja

Até agora usamos o HateBR, que já vem bem preparado (uma coluna de texto, um rótulo limpo). Bases do mundo real raramente chegam assim. Vamos usar o **B2W-Reviews01** — mais de 130 mil avaliações reais de produtos da Americanas.com, coletadas em 2018, com nota de 1 a 5 estrelas e se a pessoa recomendaria o produto a um amigo (CC BY-NC-SA 4.0).

Fonte: [github.com/b2wdigital/b2w-reviews01](https://github.com/b2wdigital/b2w-reviews01)

Essa base é genuinamente suja: tem linha sem texto, review duplicada, texto em CAIXA ALTA, preço colado no meio da frase ("R$1994.20,eu consegui..."), números, pontuação estranha. Antes de classificar qualquer coisa, precisa limpar.

In [ ]:
from sklearn.model_selection import train_test_split

url_b2w = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
b2w = pd.read_csv(url_b2w, low_memory=False)

print("Linhas:", len(b2w))
print("Colunas:", b2w.columns.tolist())
print()
print("Nulos em review_text:", b2w["review_text"].isna().sum())
print("Nulos em recommend_to_a_friend:", b2w["recommend_to_a_friend"].isna().sum())
print("Reviews duplicadas:", b2w["review_text"].duplicated().sum())
print()
print(b2w[["overall_rating", "recommend_to_a_friend", "review_text"]].sample(3, random_state=1).to_string())

In [ ]:
b2w

### Passo 1 — Limpar e preparar

O rótulo que vamos prever é `recommend_to_a_friend` ("Yes"/"No" — vira 1/0). Antes de vetorizar:

1. Tire as linhas com `review_text` ou `recommend_to_a_friend` nulos (`.dropna`).
2. Tire as reviews duplicadas (`.drop_duplicates(subset=["review_text"])`).
3. **(Sua vez)** Escreva uma função `limpar_review(texto)` usando regex que:
   - Converte pra minúsculas;
   - Troca valores em reais (`R$1994.20`, `r$ 50,00` etc.) por um marcador `" dinheiro "` (dica: `r'r\$\s*\d+[.,]?\d*'`);
   - Troca qualquer número restante por `" numero "`;
   - Remove tudo que não for letra (acentuada ou não) ou espaço;
   - Colapsa espaços duplos.
4. Aplique a função em toda a coluna `review_text` com `.apply(...)`.
5. Por que faz sentido trocar preço/número por um marcador em vez de simplesmente apagar? (Pense no que aconteceria com "R$1994.20,eu consegui comprar" se você só apagasse os dígitos sem colocar nada no lugar.)

Amostre **6.000 linhas** (`.sample(6000, random_state=1)`) antes de seguir — a base inteira é grande demais pra rodar tudo em segundos no Colab gratuito.

In [ ]:
# Sua vez — Passo 1
# b2w_limpo = b2w.dropna(subset=[...]).drop_duplicates(subset=[...])
# b2w_limpo = b2w_limpo.sample(6000, random_state=1)
#
# def limpar_review(texto):
#     ...
#
# b2w_limpo["texto_limpo"] = b2w_limpo["review_text"].apply(limpar_review)


### Passo 2 — Os três métodos, na mesma base

Agora repita a comparação de hoje, só que nesta base nova:

1. **Detector por palavra-chave** (como `contem_linguagem_inadequada` das Sessões 03-04): monte uma lista pequena de palavras claramente negativas ("péssimo", "horrível", "não recomendo"...) e classifique como "não recomenda" (0) qualquer review que contenha alguma delas; o resto vira "recomenda" (1).
2. **Léxico de sentimento** (`polaridade_frase_simples` + `lexico`, já carregados no setup): some a polaridade da review; se a soma for negativa, classifique como "não recomenda".
3. **Classificador TF-IDF + Naive Bayes**: repita o loop de unigrama / uni+bigrama / uni+bi+trigrama que já fizemos no material, agora com `X_treino`/`X_teste` vindos de `texto_limpo`.

Pra cada um, calcule a acurácia contra `recommend_to_a_friend` (convertido pra 1/0) e monte uma tabela comparando os três, igual à do material de hoje.

**Antes de comparar:** calcule também a acurácia de um "modelo" que sempre chuta a classe mais frequente (`y_treino.mode()[0]` repetido pra todo mundo). Essa base **não é balanceada** como o HateBR — vale a pena saber qual é o piso antes de comemorar qualquer acurácia.

In [ ]:
# Sua vez — Passo 2
# y = (b2w_limpo["recommend_to_a_friend"] == "Yes").astype(int)
#
# 2a. baseline "sempre a classe majoritária"
# ...
#
# 2b. detector por palavra-chave
# ...
#
# 2c. léxico de sentimento
# ...
#
# 2d. TF-IDF + Naive Bayes (uni / uni+bi / uni+bi+tri)
# ...


### Passo 3 — Reflexão

Responda com um comentário no código (não precisa de resposta "certa", mas precisa ser sua conclusão de verdade, olhando os números que você achou):

1. Qual dos três métodos ficou mais perto do teto (ML) e qual ficou mais perto do piso (baseline sempre-a-mesma-classe)?
2. Essa base tem bem mais reviews positivas (recomenda) que negativas. Isso muda como você interpreta "86% de acurácia", por exemplo? Uma acurácia alta aqui significa a mesma coisa que uma acurácia alta no HateBR (que é balanceado 50/50)?
3. Se você tivesse que escolher só **um** desses três métodos pra colocar no ar amanhã, num sistema real da loja, qual escolheria — e o que pesaria mais na decisão além da acurácia (custo computacional, interpretabilidade, precisa de dado rotulado ou não)?